# সহজ ভাষায় Notebook Guide

এই notebook-এ project-এর **একই `src/` module** (app.py আর cli.py যেগুলো ব্যবহার করে) cell-by-cell চালিয়ে CrewAI research team খুলে দেখা হয়েছে। Technical term English-এ, explanation Bangla-তে।

## কীভাবে ব্যবহার করবেন?

1. Cell উপর থেকে নিচে run করুন।
2. Live cell চালানোর আগে লিখে ফেলুন: report publish হবে নাকি escalate — আর কেন।
3. Section 5-7 real Ollama (`llama3.1:8b`) লাগে, প্রতিটা run ১-৩ মিনিট।

# Collaborative Multi-Agent Research Team — Exploration

1. Local corpus আর BM25 search
2. Code-level citation verification
3. Guardrail — task-এর output code-এ যাচাই
4. পুরো crew + Flow, fake LLM দিয়ে (offline)
5. Live: researcher নিজে search tool ডাকলে কী হয় (`tool` mode)
6. Live: grounded team (`sequential` mode)
7. Live: multi-agent বনাম single-agent

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import config  # sets CREWAI_DISABLE_TELEMETRY etc. BEFORE crewai is imported
import logging
import pandas as pd
from crewai.llms.base_llm import BaseLLM

from config import get_config
from src.crew.team import build_crew, report_guardrail, research_guardrail
from src.flow.research_flow import run_research
from src.llm.factory import ollama_status
from src.tools.corpus_search import _cached_corpus
from src.verify.citations import verify

logging.getLogger("crewai").setLevel(logging.ERROR)
pd.set_option("display.max_colwidth", 120)
cfg = get_config()
corpus = _cached_corpus(str(cfg.data.corpus_dir))
print(ollama_status())

(True, 'Ollama ready')


## 1. Local Corpus আর BM25

Research team web-এ যায় না — শুধু `data/corpus/`-এর ৮টা document পড়ে (এই course-এর verified research থেকে লেখা)। প্রতিটা section একটা passage, id `file:n` — যেমন `rag:3`। BM25 ranking stdlib-এ লেখা, embedding লাগে না।

In [2]:
print(len(corpus.passages), "passages from", len({p.doc for p in corpus.passages}), "documents")
for q in ["why do multi-agent systems fail", "interrupt resume checkpointer", "safe calculator eval"]:
    hits = corpus.search(q, k=3)
    print(f"\n{q!r}")
    for p, score in hits:
        print(f"   [{p.id}] {score:5.2f}  {p.title}")

34 passages from 8 documents

'why do multi-agent systems fail'
   [multi_agent_systems:4]  6.30  Why they fail
   [multi_agent_systems:2]  6.12  The counter-argument
   [multi_agent_systems:1]  4.71  Anthropic's research system

'interrupt resume checkpointer'
   [langgraph:3] 10.20  Persistence and human-in-the-loop

'safe calculator eval'
   [tool_calling:4] 11.44  Safe tools


## 2. Citation Verification — code, LLM না

একই llama3.1:8b দিয়ে reviewer আর judge দুটো করালে self-preference bias থাকে (LLM-as-judge গবেষণা)। তাই যা code পারে, code-এ: id আসল কিনা, notes-এ ছিল কিনা, আর বাক্যের content word cited passage-এ কতটা আছে (support score)।

In [3]:
demo = ("Independent agents amplified errors about 17 times [multi_agent_systems:3]. "
        "Bananas are a great source of potassium for athletes [multi_agent_systems:3]. "
        "This sentence makes a claim but cites nothing at all. "
        "A made-up source is cited here for this claim [rag:15].")
v = verify(demo, corpus)
display(pd.DataFrame([c.__dict__ for c in v.checks]))
print(v.summary())
print("\nfeedback for the next round:\n" + v.feedback())

,sentence,ids,support,verdict
0,Independent agents amplified errors about 17 times [multi_agent_systems:3].,[multi_agent_systems:3],1.0,supported
1,Bananas are a great source of potassium for athletes [multi_agent_systems:3].,[multi_agent_systems:3],0.0,weak
2,This sentence makes a claim but cites nothing at all.,[],0.0,uncited
3,A made-up source is cited here for this claim [rag:15].,[rag:15],0.0,invalid_id


{'sentences': 4, 'cited': 3, 'supported': 1, 'weak': 1, 'uncited': 1, 'invalid_ids': ['rag:15'], 'not_in_notes': [], 'citation_precision': 0.33}

feedback for the next round:
These citation ids do not exist: ['rag:15']. Use only ids returned by search_corpus.
Add a citation id to these factual sentences or delete them: This sentence makes a claim but cites nothing at all.
These sentences are not supported by the passage they cite; rewrite them to match the source or remove them: Bananas are a great source of potassium for athletes [multi_agent_systems:3].


## 3. Guardrail

CrewAI-র task guardrail = একটা function যেটা output দেখে `(True, output.raw)` বা `(False, কারণ)` ফেরত দেয়। False হলে কারণ সহ task আবার চলে (`guardrail_max_retries`)। দুটো ফাঁদ: file-এ `from __future__ import annotations` থাকলে guardrail-এর return annotation string হয়ে যায় আর CrewAI task বানাতেই reject করে (তাই আমাদের guardrail-এ annotation নেই); আর `(True, output.raw)` ফেরত দিন, `(True, output)` না।

In [4]:
class Out:
    def __init__(self, raw): self.raw = raw

check = research_guardrail(str(cfg.data.corpus_dir))
for raw in ["- Only one fact here [rag:1]",
            "- a [rag:1]\n- b [rag:2]\n- c [rag:15]\n- d [rag:3]",
            "- a [rag:1]\n- b [rag:2]\n- c [vector_databases:1]\n- d [rag:3]"]:
    print(check(Out(raw)))

(False, 'Give at least 4 bullet points, each ending with a citation id like [rag:2], using at least 3 different ids. Call search_corpus again if you need more facts.')
(False, "Unknown citation ids ['rag:15']. Only cite ids shown by search_corpus, like [rag:2].")
(True, '- a [rag:1]\n- b [rag:2]\n- c [vector_databases:1]\n- d [rag:3]')


## 4. পুরো Crew + Flow — fake LLM দিয়ে

একটা scripted `BaseLLM` CrewAI-র ReAct format-এ ("Thought: ... Final Answer: ...") উত্তর দেয়। তাই crew, guardrail retry, Flow-এর router — সব সত্যিই চলে, শুধু model নকল। প্রথম উত্তরটা ইচ্ছে করে ভুল (মাত্র একটা fact) — guardrail ফিরিয়ে দেয়।

In [5]:
class FakeLLM(BaseLLM):
    def __init__(self, answers):
        super().__init__(model="fake")
        self._answers, self.calls = list(answers), 0
    def call(self, messages, *a, **k):
        self.calls += 1
        ans = self._answers.pop(0) if len(self._answers) > 1 else self._answers[0]
        return f"Thought: I now know the final answer\nFinal Answer: {ans}"
    def supports_function_calling(self):
        return False

NOTES = ("- MAST grouped 14 failure modes into three categories [multi_agent_systems:4]\n"
         "- Adding a verification step was one of the most effective fixes [multi_agent_systems:4]\n"
         "- Independent agents amplified errors about 17 times [multi_agent_systems:3]\n"
         "- Agent loops need a maximum number of steps and repeat detection [react_agents:3]")
REPORT = ("# Why Multi-Agent Systems Fail\n\n## Failure modes\nThe MAST taxonomy grouped 14 failure modes into "
          "system design, inter-agent misalignment and task verification [multi_agent_systems:4]. Independent agents "
          "amplified errors about 17 times versus about 4 times with a central orchestrator [multi_agent_systems:3].\n\n"
          "## Fixes\nAdding a verification step was one of the most effective fixes [multi_agent_systems:4]. "
          "Agent loops need hard stop conditions such as a maximum number of steps [react_agents:3].")

fake = FakeLLM(["- too short [rag:1]", NOTES, REPORT, REPORT])
state = run_research("Why multi-agent systems fail", llm=fake)
print("LLM calls:", fake.calls, "(the first research answer was rejected by the guardrail)")
print("outcome:", state.outcome, "| rounds:", state.rounds, "| retrieved:", state.passages)
print(state.verification)

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: ResearchFlow                                                                                             │
│  ID: 0c6143dd-9fb7-4534-ad95-577eabd41a34                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.29                                                                                       │
│  Latest version:  1.15.30                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: ResearchFlow                                                                                             │
│  ID: 0c6143dd-9fb7-4534-ad95-577eabd41a34                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: retrieve                                                                                               │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: retrieve                                                                                               │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: gate                                                                                                   │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: publish                                                                                                │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: publish                                                                                                │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: ResearchFlow                                                                                             │
│  ID: 0c6143dd-9fb7-4534-ad95-577eabd41a34                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

LLM calls: 4 (the first research answer was rejected by the guardrail)
outcome: published | rounds: 1 | retrieved: ['multi_agent_systems:4', 'multi_agent_systems:1', 'react_agents:3', 'multi_agent_systems:2', 'vector_databases:3', 'multi_agent_systems:3']
{'sentences': 4, 'cited': 4, 'supported': 4, 'weak': 0, 'uncited': 0, 'invalid_ids': [], 'not_in_notes': [], 'citation_precision': 1.0}


## 5. Live — researcher নিজে search tool ডাকলে (`tool` mode)

Framework-এর "আসল" multi-agent ধারণা: researcher agent নিজে `search_corpus` tool ডাকবে। দেখি llama3.1:8b আসলে কী করে। (আমাদের আগের live run-এ model tool **একবারও ডাকেনি** — tool-call JSON text হিসেবে লিখে নিজেই "search result" আর id বানিয়েছিল।)

In [6]:
TOPIC = "Why multi-agent systems fail and how to make them reliable"
outputs = []
state = run_research(TOPIC, "tool", on_task=outputs.append)
print("outcome:", state.outcome, "| rounds:", state.rounds, "|", state.seconds, "s")
print("verification:", state.verification)
if outputs:
    print("\nResearcher output (first 900 chars):\n" + outputs[0].raw[:900])

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: ResearchFlow                                                                                             │
│  ID: 156ac148-9b96-419f-ac65-84c9db119212                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: ResearchFlow                                                                                             │
│  ID: 156ac148-9b96-419f-ac65-84c9db119212                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: retrieve                                                                                               │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: retrieve                                                                                               │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: run_team                                                                                               │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: gate                                                                                                   │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: needs_human                                                                                            │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: needs_human                                                                                            │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: ResearchFlow                                                                                             │
│  ID: 156ac148-9b96-419f-ac65-84c9db119212                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

outcome: needs_human | rounds: 2 | 135.4 s
verification: {'error': 'Exception: Task failed guardrail validation after 2 retries. Last error: Give at least 4 bullet points, each ending with a citation id like [rag:2], using at least 3 different ids. Call search_corpus again if you need more facts.'}


## 6. Live — Grounded Team (`sequential`, default)

এবার retrieval code করে (BM25), passage-গুলো researcher-এর task-এ বসে। Agent-দের কাজ বাছাই, লেখা, সম্পাদনা — আর শেষে code citation যাচাই করে।

In [7]:
outputs = []
state = run_research(TOPIC, "sequential", on_task=outputs.append)
print("outcome:", state.outcome, "| rounds:", state.rounds, "|", state.seconds, "s | retrieved:", state.passages)
print("verification:", state.verification)
print("tokens:", state.token_usage)
print("\n" + state.report)

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: ResearchFlow                                                                                             │
│  ID: 8feaadd0-b0a6-4d04-bf71-f38851898b90                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: ResearchFlow                                                                                             │
│  ID: 8feaadd0-b0a6-4d04-bf71-f38851898b90                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: retrieve                                                                                               │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: retrieve                                                                                               │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: run_team                                                                                               │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: gate                                                                                                   │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: publish                                                                                                │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: publish                                                                                                │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: ResearchFlow                                                                                             │
│  ID: 8feaadd0-b0a6-4d04-bf71-f38851898b90                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

outcome: published | rounds: 1 | 99.2 s | retrieved: ['multi_agent_systems:2', 'multi_agent_systems:4', 'multi_agent_systems:1', 'react_agents:3', 'react_agents:4', 'vector_databases:3']
verification: {'sentences': 11, 'cited': 7, 'supported': 7, 'weak': 0, 'uncited': 4, 'invalid_ids': [], 'not_in_notes': [], 'citation_precision': 1.0}
tokens: {'total_tokens': 8112, 'prompt_tokens': 5505, 'cached_prompt_tokens': 243, 'completion_tokens': 2607, 'reasoning_tokens': 0, 'cache_creation_tokens': 0, 'successful_requests': 9}

# Why Multi-Agent Systems Fail and How to Make Them Reliable

## Common Failure Modes in Multi-Agent Systems

Multi-agent systems often fail due to agents not sharing full context, leading to conflicting implicit decisions [multi_agent_systems:2]. The most common single failure modes in multi-agent systems are step repetition, reasoning-action mismatch, and not knowing when to stop [multi_agent_systems:4].

## Effective Fixes for Multi-Agent System Failures

Adding a ve

In [8]:
for o in outputs:
    print(f"===== {o.agent} =====\n{o.raw[:700]}\n")

===== Senior Research Specialist =====
Here are the facts relevant to Why multi-agent systems fail and how to make them reliable:

• Agents that do not share full context make conflicting implicit decisions, leading to failure [multi_agent_systems:2].
• The most common single failure modes in multi-agent systems are step repetition, reasoning-action mismatch, and not knowing when to stop [multi_agent_systems:4].
• Adding a verification step is one of the most effective fixes for multi-agent system failures [multi_agent_systems:4].
• Agent loops need hard stop conditions, including a maximum number of steps, detection of repeated identical tool calls, a deadline, and a budget for consecutive invalid calls [react_agents:3].
• Tempe

===== Technical Report Writer =====
# Why Multi-Agent Systems Fail and How to Make Them Reliable

## Common Failure Modes in Multi-Agent Systems

Multi-agent systems often fail due to agents not sharing full context, leading to conflicting implicit decisions 

## 7. Multi-Agent বনাম Single-Agent

Anthropic-এর multi-agent system single agent-কে ৯০% হারিয়েছিল — কিন্তু ~১৫ গুণ token খরচ করে, parallel subagent দিয়ে। আমাদের team sequential, একটাই local model। একই passage আর একই guardrail দিয়ে single agent কেমন করে? তিনটা topic-এ মাপি।

In [9]:
rows = []
for topic in cfg.app.topics[1:4]:
    for mode in ("sequential", "single"):
        s = run_research(topic, mode)
        v = s.verification
        rows.append({"topic": topic[:40], "mode": mode, "outcome": s.outcome, "rounds": s.rounds,
                     "supported/cited": f"{v.get('supported', 0)}/{v.get('cited', 0)}",
                     "uncited": v.get("uncited"), "precision": v.get("citation_precision"),
                     "seconds": s.seconds, "tokens": s.token_usage.get("total_tokens")})
comparison = pd.DataFrame(rows)
comparison

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: ResearchFlow                                                                                             │
│  ID: df65f36b-3b8c-4180-b79f-19d6a13c2d3f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: ResearchFlow                                                                                             │
│  ID: df65f36b-3b8c-4180-b79f-19d6a13c2d3f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: retrieve                                                                                               │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: retrieve                                                                                               │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: run_team                                                                                               │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: gate                                                                                                   │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: publish                                                                                                │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: publish                                                                                                │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: ResearchFlow                                                                                             │
│  ID: e92c070c-27ad-4dea-b3a9-216692cf6179                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

,topic,mode,outcome,rounds,supported/cited,uncited,precision,seconds,tokens
0,How retrieval-augmented generation groun,sequential,published,1,7/7,1,1.00,79.2,6381
1,How retrieval-augmented generation groun,single,published,1,7/7,7,1.00,37.7,1018
2,Safe tool calling with local models,sequential,published,1,8/8,2,1.00,97.4,7992
3,Safe tool calling with local models,single,published,2,15/15,2,1.00,104.0,1257
4,Self-correcting code generation,sequential,published,1,6/7,2,0.86,99.4,8022
5,Self-correcting code generation,single,published,1,7/7,3,1.00,38.9,952


In [10]:
comparison.groupby("mode")[["seconds", "tokens", "precision"]].mean().round(2)

,seconds,tokens,precision
mode,,,
sequential,92.0,7465.00,0.95
single,60.2,1075.67,1.00


**কী শিখলাম:**

* ছোট local model-এ CrewAI-র ভেতরের tool calling নির্ভরযোগ্য না — model search ডাকার বদলে result বানিয়ে নিতে পারে। Retrieval code-এ সরালে সেই hallucination-এর পথ বন্ধ হয়।
* Code-level verification (citation id + support score) হলো এই project-এর আসল quality gate — MAST-এর পর্যবেক্ষণ মিলে যায়: verification step সবচেয়ে কার্যকর fix-গুলোর একটা।
* Multi-agent আমাদের sequential setup-এ বেশি সময় আর token খরচ করে; quality-তে লাভ আছে কিনা উপরের table নিজে বিচার করুন — এটাই Cognition আর Anthropic-এর বিতর্কের ছোট সংস্করণ।
* Live সংখ্যা run থেকে run-এ বদলায় — একটা run দেখে সিদ্ধান্ত না।

## 🧠 Brain Teasers

1. `max_uncited_ratio`-কে 0.2 করুন। কয়টা run escalate হয়? Uncited বাক্যগুলো আসলে fact নাকি transition?
2. Section 5-এর `tool` mode-এ researcher-এর `max_iter` 2 করুন। Output কি বদলায়?
3. Corpus-এ একটা নতুন document যোগ করুন (`data/corpus/evaluation.md`) — BM25 কি সেটা খুঁজে পায়, আর report-এ citation আসে?
4. Reviewer agent বাদ দিয়ে (২-agent crew) Section 7 চালান। Verification সংখ্যা খারাপ হয়, নাকি শুধু সময় বাঁচে?